In [ ]:
import sys
import subprocess

print("Installing dependencies (NVIDIA PhysicsNeMo, PyTorch Autograd, Scikit-Learn, psutil)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "nvidia-physicsnemo", "scikit-learn", "matplotlib", "pandas", "numpy", "scipy", "psutil"
])

import os
import glob
import time
import json
import psutil
import torch
import numpy as np
import pandas as pd

import physicsnemo
from physicsnemo.models.mlp.fully_connected import FullyConnected

BASE_PATH = os.getcwd()

DATASET_DIR = os.path.join(BASE_PATH, "dataset")
MODEL_DIR = os.path.join(BASE_PATH, "models")
FIG_DIR = os.path.join(BASE_PATH, "figures")
ABLATION_DIR = os.path.join(BASE_PATH, "ablation_results")
METRICS_DIR = os.path.join(BASE_PATH, "research_paper_metrics")

for d in [DATASET_DIR, MODEL_DIR, FIG_DIR, ABLATION_DIR, METRICS_DIR]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

system_diagnostics = {
    "python_version": sys.version.split()[0],
    "pytorch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "device_count": torch.cuda.device_count(),
    "gpu_total_memory_mb": round(torch.cuda.get_device_properties(0).total_memory / (1024**2), 2) if torch.cuda.is_available() else 0,
    "cpu_count_logical": psutil.cpu_count(logical=True),
    "cpu_count_physical": psutil.cpu_count(logical=False),
    "system_ram_gb": round(psutil.virtual_memory().total / (1024**3), 2)
}

sys_log_path = os.path.join(METRICS_DIR, "system_hardware_environment_spec.json")
with open(sys_log_path, "w") as f:
    json.dump(system_diagnostics, f, indent=4)

print(f"Target Directory  : {BASE_PATH}")
print(f"Execution Device  : {system_diagnostics['device_name']} (CUDA {system_diagnostics['cuda_version']})")
print(f"VRAM Capacity     : {system_diagnostics['gpu_total_memory_mb']} MB")
print(f"System RAM        : {system_diagnostics['system_ram_gb']} GB | CPU Cores: {system_diagnostics['cpu_count_logical']}")


In [ ]:
t_data_start = time.perf_counter()

all_csvs = glob.glob(os.path.join(DATASET_DIR, "*.csv")) + glob.glob(os.path.join(BASE_PATH, "*.csv")) + glob.glob(os.path.join(BASE_PATH, "**", "*.csv"), recursive=True)

all_csvs = list(dict.fromkeys(all_csvs))
ignore_kw = ["summary", "matrix", "history", "metrics", "eval", "outliers", "ablation", "paper", "system"]
candidate_csvs = [f for f in all_csvs if not any(kw in os.path.basename(f).lower() for kw in ignore_kw)]

DATASET_CSV = None
for csv_path in sorted(candidate_csvs, key=os.path.getmtime, reverse=True):
    try:
        temp_df = pd.read_csv(csv_path, nrows=5)
        if any(col.startswith("thickness_pt_") for col in temp_df.columns):
            DATASET_CSV = csv_path
            break
    except Exception:
        continue

if DATASET_CSV is None:
    raise FileNotFoundError(f"Could not locate raw spatial dataset CSV inside '{BASE_PATH}'.")

dataset_file_size_bytes = os.path.getsize(DATASET_CSV)
dataset_file_size_mb = round(dataset_file_size_bytes / (1024**2), 4)

df_full = pd.read_csv(DATASET_CSV)
N_TOTAL = len(df_full)

FEATURE_COLS = ["temperature_C", "pulse_time_s", "num_cycles", "tma_mole_fraction"]
TARGET_COLS = [col for col in df_full.columns if col.startswith("thickness_pt_")]
N_SPATIAL_POINTS = len(TARGET_COLS)

np.random.seed(42)
torch.manual_seed(42)

all_indices = np.random.permutation(N_TOTAL)
N_TEST = min(2000, int(0.2 * N_TOTAL))
test_idx = all_indices[:N_TEST]
pool_idx = all_indices[N_TEST:]

df_test = df_full.iloc[test_idx].reset_index(drop=True)
df_pool = df_full.iloc[pool_idx].reset_index(drop=True)

X_pool_raw = df_pool[FEATURE_COLS].values.astype(np.float32)
Y_pool_raw = df_pool[TARGET_COLS].values.astype(np.float32)

X_mean, X_std = X_pool_raw.mean(0), X_pool_raw.std(0) + 1e-8
Y_mean, Y_std = Y_pool_raw.mean(0), Y_pool_raw.std(0) + 1e-8

X_test_norm = (df_test[FEATURE_COLS].values.astype(np.float32) - X_mean) / X_std
Y_test_nm = df_test[TARGET_COLS].values.astype(np.float32)
X_test_tensor = torch.tensor(X_test_norm, device=device)

x_test_mem_kb = round(X_test_tensor.element_size() * X_test_tensor.nelement() / 1024, 2)
y_test_mem_kb = round(Y_test_nm.nbytes / 1024, 2)

t_data_end = time.perf_counter()

dataset_ingestion_log = {
    "raw_dataset_filename": os.path.basename(DATASET_CSV),
    "raw_dataset_file_size_bytes": dataset_file_size_bytes,
    "raw_dataset_file_size_mb": dataset_file_size_mb,
    "total_samples": N_TOTAL,
    "spatial_target_coordinates": N_SPATIAL_POINTS,
    "holdout_test_samples": N_TEST,
    "training_pool_samples": len(df_pool),
    "x_test_tensor_memory_kb": x_test_mem_kb,
    "y_test_array_memory_kb": y_test_mem_kb,
    "ingestion_and_split_time_seconds": round(t_data_end - t_data_start, 4)
}

meta_bundle = {
    "X_mean": X_mean, "X_std": X_std,
    "Y_mean": Y_mean, "Y_std": Y_std,
    "FEATURE_COLS": FEATURE_COLS,
    "TARGET_COLS": TARGET_COLS,
    "N_SPATIAL_POINTS": N_SPATIAL_POINTS,
    "dataset_ingestion_log": dataset_ingestion_log
}
torch.save(meta_bundle, os.path.join(ABLATION_DIR, "ablation_metadata_bundle.pt"))

print(f"Raw CSV File Path  : {DATASET_CSV}")
print(f"Raw CSV File Size  : {dataset_file_size_mb} MB ({dataset_file_size_bytes:,} bytes)")
print(f"Total Rows x Cols  : {N_TOTAL} samples x {df_full.shape[1]} columns")
print(f"Holdout Test Size  : {N_TEST} samples ({x_test_mem_kb + y_test_mem_kb:.2f} KB in memory)")
print(f"Training Pool Size : {len(df_pool)} samples")
print(f"Ingestion Runtime  : {dataset_ingestion_log['ingestion_and_split_time_seconds']} seconds")


In [ ]:
import os
import time
import copy
import torch
import numpy as np
import pandas as pd

DATA_TIERS = [20, 100, 1000, min(8000, len(df_pool))]
PHYSICS_LEVELS = ["P0", "P1", "P2", "P3"]
N_EPOCHS = 2500
PATIENCE = 300
LAMBDA_PHYS = 0.5

spatial_weights = torch.ones(N_SPATIAL_POINTS, device=device)
spatial_weights[6:40] = 1.5
spatial_weights[40:61] = 3.0
spatial_weights[61:94] = 1.5

def weighted_mse_loss(pred, target):
    return torch.mean(spatial_weights * (pred - target) ** 2)

boundary_w = max(1, int(0.05 * N_SPATIAL_POINTS))
top_indices = list(range(0, boundary_w)) + list(range(N_SPATIAL_POINTS - boundary_w, N_SPATIAL_POINTS))

X_mean_t = torch.tensor(X_mean, device=device)
X_std_t = torch.tensor(X_std, device=device)
Y_mean_t = torch.tensor(Y_mean, device=device)
Y_std_t = torch.tensor(Y_std, device=device)

training_performance_records = []
timing_log_csv_path = os.path.join(METRICS_DIR, "training_and_inference_timings.csv")

if os.path.exists(timing_log_csv_path):
    try:
        prev_df = pd.read_csv(timing_log_csv_path)
        training_performance_records = prev_df.to_dict("records")
    except Exception:
        pass

print(f"STARTING 16-MODEL FACTORIAL ABLATION MATRIX ({len(PHYSICS_LEVELS)} Levels x {len(DATA_TIERS)} Tiers)")

for p_level in PHYSICS_LEVELS:
    for n_samples in DATA_TIERS:
        model_name = f"model_{p_level}_N{n_samples}"
        ckpt_path = os.path.join(MODEL_DIR, f"{model_name}_checkpoint.pt")
        final_path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")

        if os.path.exists(final_path):
            print(f"[SKIP] {model_name} is already fully trained and serialized.")
            continue

        print(f"\n>>> Training {model_name} (Physics: {p_level} | Training Samples: {n_samples}) <<<")

        df_sub = df_pool.iloc[:n_samples].copy()
        n_train = max(1, int(0.8 * n_samples))

        X_sub_raw = df_sub[FEATURE_COLS].values.astype(np.float32)
        Y_sub_raw = df_sub[TARGET_COLS].values.astype(np.float32)

        X_sub_norm = (X_sub_raw - X_mean) / X_std
        Y_sub_norm = (Y_sub_raw - Y_mean) / Y_std

        X_tr = torch.tensor(X_sub_norm[:n_train], device=device, requires_grad=True)
        Y_tr = torch.tensor(Y_sub_norm[:n_train], device=device)

        val_end = len(X_sub_norm) if n_samples > 20 else n_train
        X_va = torch.tensor(X_sub_norm[n_train:val_end] if val_end > n_train else X_sub_norm[:n_train], device=device)
        Y_va = torch.tensor(Y_sub_norm[n_train:val_end] if val_end > n_train else Y_sub_norm[:n_train], device=device)

        GPC_tr = torch.tensor(df_sub["gpc_nm_per_cycle"].values[:n_train].astype(np.float32), device=device)

        with torch.no_grad():
            N_cyc_tr = X_tr[:, 2] * X_std_t[2] + X_mean_t[2]
            linearity_mask = torch.clamp(N_cyc_tr / 10.0, min=0.0, max=1.0).detach()

        surrogate = FullyConnected(
            in_features=len(FEATURE_COLS),
            out_features=N_SPATIAL_POINTS,
            num_layers=6,
            layer_size=128,
        ).to(device)

        optimizer = torch.optim.Adam(surrogate.parameters(), lr=2e-3, weight_decay=5e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

        start_epoch = 0
        best_val_loss = float('inf')
        epochs_no_improve = 0
        best_state = None

        if os.path.exists(ckpt_path):
            print(f"Resuming {model_name} from active checkpoint...")
            ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
            surrogate.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            start_epoch = ckpt["epoch"] + 1
            best_val_loss = ckpt["best_val_loss"]
            best_state = ckpt["best_state"]

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        t_train_start = time.perf_counter()

        for epoch in range(start_epoch, N_EPOCHS):
            surrogate.train()
            optimizer.zero_grad()

            pred_norm = surrogate(X_tr)

            if p_level == "P3":
                loss_data = weighted_mse_loss(pred_norm, Y_tr)
            else:
                loss_data = torch.nn.MSELoss()(pred_norm, Y_tr)

            loss_physics = torch.tensor(0.0, device=device)

            if p_level in ["P1", "P2", "P3"]:
                X_zero = X_tr.clone().detach()
                X_zero[:, 2] = float((0.0 - X_mean[2]) / X_std[2])
                pred_zero = surrogate(X_zero)
                T_zero = pred_zero * Y_std_t + Y_mean_t
                loss_origin = torch.mean(T_zero ** 2)
                loss_physics = loss_physics + loss_origin

            if p_level in ["P2", "P3"]:
                pred = pred_norm * Y_std_t + Y_mean_t
                T_top = pred[:, top_indices].mean(dim=1)
                dTtop_dX = torch.autograd.grad(
                    outputs=T_top, inputs=X_tr,
                    grad_outputs=torch.ones_like(T_top),
                    create_graph=True, retain_graph=True
                )[0]
                dTtop_dN = dTtop_dX[:, 2] / X_std_t[2]
                loss_kinetics = torch.mean((dTtop_dN - GPC_tr) ** 2)
                loss_physics = loss_physics + loss_kinetics

            if p_level == "P3":
                pred = pred_norm * Y_std_t + Y_mean_t
                T_mean_global = pred.mean(dim=1)
                dTglobal_dX = torch.autograd.grad(
                    outputs=T_mean_global, inputs=X_tr,
                    grad_outputs=torch.ones_like(T_mean_global),
                    create_graph=True, retain_graph=True
                )[0]
                dTglobal_dN = dTglobal_dX[:, 2] / X_std_t[2]
                d2Tglobal_dX2 = torch.autograd.grad(
                    outputs=dTglobal_dN, inputs=X_tr,
                    grad_outputs=torch.ones_like(dTglobal_dN),
                    create_graph=True, retain_graph=True
                )[0]
                d2Tglobal_dN2 = d2Tglobal_dX2[:, 2] / X_std_t[2]
                loss_linear = torch.mean(linearity_mask * (d2Tglobal_dN2 ** 2))

                laplacian = pred[:, 2:] - 2 * pred[:, 1:-1] + pred[:, :-2]
                loss_spatial = torch.mean(laplacian ** 2)

                loss_physics = loss_physics + loss_linear + (0.1 * loss_spatial)

            loss = loss_data + (LAMBDA_PHYS * loss_physics)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(surrogate.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            surrogate.eval()
            with torch.no_grad():
                val_pred = surrogate(X_va)
                val_loss = torch.nn.MSELoss()(val_pred, Y_va).item()

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                best_state = copy.deepcopy(surrogate.state_dict())
            else:
                epochs_no_improve += 1

            if (epoch + 1) % 500 == 0 or epoch == N_EPOCHS - 1:
                print(f"  Epoch {epoch+1:4d}/{N_EPOCHS} | Train Loss: {loss.item():.6f} | Val Loss: {val_loss:.6f}")

                torch.save({
                    "epoch": epoch,
                    "model_state_dict": surrogate.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_val_loss": best_val_loss,
                    "best_state": best_state
                }, ckpt_path)

            if epochs_no_improve >= PATIENCE:
                print(f"  Early stopping triggered at Epoch {epoch+1}.")
                break

        t_train_end = time.perf_counter()
        total_train_sec = round(t_train_end - t_train_start, 4)
        epochs_executed = (epoch + 1) - start_epoch
        avg_ms_per_epoch = round((total_train_sec / max(1, epochs_executed)) * 1000, 3)

        peak_vram_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 2) if torch.cuda.is_available() else 0

        if best_state is not None:
            surrogate.load_state_dict(best_state)

        torch.save({
            "model_name": model_name,
            "physics_level": p_level,
            "training_samples": n_samples,
            "model_state_dict": surrogate.state_dict(),
            "X_mean": X_mean, "X_std": X_std,
            "Y_mean": Y_mean, "Y_std": Y_std
        }, final_path)

        final_file_size_bytes = os.path.getsize(final_path)
        final_file_size_kb = round(final_file_size_bytes / 1024, 2)

        surrogate.eval()
        t_inf_start = time.perf_counter()
        with torch.no_grad():
            _ = surrogate(X_test_tensor).cpu().numpy()
        t_inf_end = time.perf_counter()

        test_inference_sec = round(t_inf_end - t_inf_start, 6)
        throughput_samples_per_sec = round(N_TEST / max(test_inference_sec, 1e-6), 2)
        latency_ms_per_sample = round((test_inference_sec / N_TEST) * 1000, 5)

        record = {
            "model_name": model_name,
            "physics_level": p_level,
            "training_samples": n_samples,
            "epochs_completed": epoch + 1,
            "early_stopping_triggered": epochs_no_improve >= PATIENCE,
            "total_train_time_seconds": total_train_sec,
            "total_train_time_minutes": round(total_train_sec / 60, 3),
            "avg_ms_per_epoch": avg_ms_per_epoch,
            "peak_vram_allocated_mb": peak_vram_mb,
            "checkpoint_size_bytes": final_file_size_bytes,
            "checkpoint_size_kb": final_file_size_kb,
            "test_set_inference_time_seconds": test_inference_sec,
            "inference_throughput_samples_per_sec": throughput_samples_per_sec,
            "single_sample_latency_ms": latency_ms_per_sample
        }
        training_performance_records.append(record)

        pd.DataFrame(training_performance_records).to_csv(timing_log_csv_path, index=False)
        print(f"[SAVED METRICS] {model_name}: Train Time = {total_train_sec}s | Throughput = {throughput_samples_per_sec} sam/s | Size = {final_file_size_kb} KB")



In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

ablation_results = []

for p_level in PHYSICS_LEVELS:
    for n_samples in DATA_TIERS:
        model_name = f"model_{p_level}_N{n_samples}"
        final_path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")

        if not os.path.exists(final_path):
            print(f"[ERROR] Missing model file: {final_path}")
            continue

        checkpoint = torch.load(final_path, map_location=device, weights_only=False)

        surrogate = FullyConnected(
            in_features=len(FEATURE_COLS),
            out_features=N_SPATIAL_POINTS,
            num_layers=6,
            layer_size=128,
        ).to(device)

        surrogate.load_state_dict(checkpoint["model_state_dict"])
        surrogate.eval()

        t_eval_start = time.perf_counter()
        with torch.no_grad():
            Y_pred_norm = surrogate(X_test_tensor).cpu().numpy()
        t_eval_end = time.perf_counter()

        eval_duration_sec = round(t_eval_end - t_eval_start, 6)

        Y_pred_nm = Y_pred_norm * Y_std + Y_mean
        abs_err = np.abs(Y_test_nm - Y_pred_nm)

        global_r2 = float(r2_score(Y_test_nm.flatten(), Y_pred_nm.flatten()))
        global_mae = float(abs_err.mean())
        global_rmse = float(np.sqrt(mean_squared_error(Y_test_nm, Y_pred_nm)))
        global_max_err = float(abs_err.max())
        global_mape = float(np.mean(abs_err / np.clip(Y_test_nm, 1e-6, None)) * 100.0)
        pct_95_err = float(np.percentile(abs_err, 95))

        top_pts = list(range(0, boundary_w)) + list(range(N_SPATIAL_POINTS - boundary_w, N_SPATIAL_POINTS))
        bot_pts = list(range(int(0.40 * N_SPATIAL_POINTS), int(0.60 * N_SPATIAL_POINTS)))

        true_top = Y_test_nm[:, top_pts].mean(axis=1)
        pred_top = Y_pred_nm[:, top_pts].mean(axis=1)
        true_bot = Y_test_nm[:, bot_pts].mean(axis=1)
        pred_bot = Y_pred_nm[:, bot_pts].mean(axis=1)

        true_conf = true_bot / np.clip(true_top, 1e-6, None)
        pred_conf = pred_bot / np.clip(pred_top, 1e-6, None)

        conf_r2 = float(r2_score(true_conf, pred_conf))
        conf_mae = float(mean_absolute_error(true_conf, pred_conf))
        conf_rmse = float(np.sqrt(mean_squared_error(true_conf, pred_conf)))

        ablation_results.append({
            "model_name": model_name,
            "physics_level": p_level,
            "training_samples": n_samples,
            "eval_duration_sec": eval_duration_sec,
            "global_spatial_r2": global_r2,
            "global_spatial_mae_nm": global_mae,
            "global_spatial_rmse_nm": global_rmse,
            "global_spatial_max_error_nm": global_max_err,
            "global_spatial_mape_pct": global_mape,
            "error_tail_95pct_nm": pct_95_err,
            "conformality_r2": conf_r2,
            "conformality_mae": conf_mae,
            "conformality_rmse": conf_rmse
        })

df_ablation = pd.DataFrame(ablation_results)
res_csv_path = os.path.join(ABLATION_DIR, "ablation_matrix_results.csv")
df_ablation.to_csv(res_csv_path, index=False)

print("\nFactorial Evaluation Summary:")
print(df_ablation[["physics_level", "training_samples", "global_spatial_r2", "global_spatial_mae_nm", "conformality_r2", "eval_duration_sec"]].to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12), gridspec_kw={'hspace': 0.3, 'wspace': 0.25})

colors = {"P0": "#d62728", "P1": "#ff7f0e", "P2": "#1f77b4", "P3": "#2ca02c"}
markers = {"P0": "o", "P1": "s", "P2": "^", "P3": "D"}
labels = {
    "P0": "P0: Pure Data-Driven MLP",
    "P1": "P1: Boundary Anchored PINN",
    "P2": "P2: Kinetics-Informed PINN",
    "P3": "P3: Multiscale Spatial PINN"
}

for p in PHYSICS_LEVELS:
    sub = df_ablation[df_ablation["physics_level"] == p]
    axes[0, 0].plot(sub["training_samples"], sub["global_spatial_r2"],
                    marker=markers[p], color=colors[p], label=labels[p], linewidth=2.0)

axes[0, 0].set_xscale("log")
axes[0, 0].set_xlabel("Training Dataset Size (N)", fontweight='bold')
axes[0, 0].set_ylabel("Global Spatial R² Score", fontweight='bold')
axes[0, 0].set_title("Ablation Curve: Global Spatial R² vs Data Volume", fontweight='bold')
axes[0, 0].grid(True, which="both", alpha=0.3)
axes[0, 0].legend()

for p in PHYSICS_LEVELS:
    sub = df_ablation[df_ablation["physics_level"] == p]
    axes[0, 1].plot(sub["training_samples"], sub["global_spatial_mae_nm"],
                    marker=markers[p], color=colors[p], label=labels[p], linewidth=2.0)

axes[0, 1].set_xscale("log")
axes[0, 1].set_yscale("log")
axes[0, 1].set_xlabel("Training Dataset Size (N)", fontweight='bold')
axes[0, 1].set_ylabel("Mean Absolute Error (nm Log Scale)", fontweight='bold')
axes[0, 1].set_title("Ablation Curve: Spatial MAE vs Data Volume", fontweight='bold')
axes[0, 1].grid(True, which="both", alpha=0.3)

for p in PHYSICS_LEVELS:
    sub = df_ablation[df_ablation["physics_level"] == p]
    axes[1, 0].plot(sub["training_samples"], sub["conformality_r2"],
                    marker=markers[p], color=colors[p], label=labels[p], linewidth=2.0)

axes[1, 0].set_xscale("log")
axes[1, 0].set_xlabel("Training Dataset Size (N)", fontweight='bold')
axes[1, 0].set_ylabel("Conformality R² Score", fontweight='bold')
axes[1, 0].set_title("Physical Integrity: Conformality Ratio R²", fontweight='bold')
axes[1, 0].grid(True, which="both", alpha=0.3)

pivot_r2 = df_ablation.pivot(index="physics_level", columns="training_samples", values="global_spatial_r2")
im = axes[1, 1].imshow(pivot_r2.values, cmap="Blues", aspect="auto")

axes[1, 1].set_xticks(range(len(DATA_TIERS)))
axes[1, 1].set_xticklabels([str(n) for n in DATA_TIERS])
axes[1, 1].set_yticks(range(len(PHYSICS_LEVELS)))
axes[1, 1].set_yticklabels(PHYSICS_LEVELS)

for i in range(len(PHYSICS_LEVELS)):
    for j in range(len(DATA_TIERS)):
        val = pivot_r2.values[i, j]
        axes[1, 1].text(j, i, f"{val:.4f}", ha="center", va="center",
                        color="white" if val > pivot_r2.values.mean() else "black", fontweight="bold")

axes[1, 1].set_xlabel("Training Dataset Size (N)", fontweight='bold')
axes[1, 1].set_ylabel("Physics Constraint Level", fontweight='bold')
axes[1, 1].set_title("4x4 Factorial Matrix: Test R² Heatmap", fontweight='bold')
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.suptitle("Physics vs. Data Volume Ablation Study (ALD FinFET Digital Twin)", fontsize=16, fontweight='bold', y=0.98)
fig_path = os.path.join(FIG_DIR, "factorial_ablation_study_master.png")
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.close()

master_fig_bytes = os.path.getsize(fig_path)
master_fig_kb = round(master_fig_bytes / 1024, 2)

print(f"[SAVED FIGURE] {fig_path} ({master_fig_kb} KB | {master_fig_bytes:,} bytes)")


In [ ]:
from scipy import stats

RAW_PREDS_DIR = os.path.join(METRICS_DIR, "raw_prediction_arrays")
POINTWISE_DIR = os.path.join(METRICS_DIR, "spatial_pointwise_profiles")
TAILS_DIR = os.path.join(METRICS_DIR, "error_tail_outliers")
TABLES_DIR = os.path.join(METRICS_DIR, "master_tables_and_latex")

for d in [RAW_PREDS_DIR, POINTWISE_DIR, TAILS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

fin_peak_pts = list(range(0, boundary_w)) + list(range(N_SPATIAL_POINTS - boundary_w, N_SPATIAL_POINTS))
sidewall_pts = list(range(boundary_w, int(0.40 * N_SPATIAL_POINTS))) + list(range(int(0.60 * N_SPATIAL_POINTS), N_SPATIAL_POINTS - boundary_w))
trench_bottom_pts = list(range(int(0.40 * N_SPATIAL_POINTS), int(0.60 * N_SPATIAL_POINTS)))
corner_pts = list(range(int(0.38 * N_SPATIAL_POINTS), int(0.42 * N_SPATIAL_POINTS))) + list(range(int(0.58 * N_SPATIAL_POINTS), int(0.62 * N_SPATIAL_POINTS)))

true_top = Y_test_nm[:, fin_peak_pts].mean(axis=1)
true_bot = Y_test_nm[:, trench_bottom_pts].mean(axis=1)
true_conf = true_bot / np.clip(true_top, 1e-6, None)

master_rows = []
all_predictions_dict = {"Y_true_test_nm": Y_test_nm}
pointwise_rows = []

df_sample_master = df_test[FEATURE_COLS + ["gpc_nm_per_cycle", "conformality_bottom_to_top"]].copy()
df_sample_master.rename(columns={"conformality_bottom_to_top": "true_conformality"}, inplace=True)
df_sample_master["true_top_thickness_nm"] = true_top
df_sample_master["true_bottom_thickness_nm"] = true_bot

timing_csv_path = os.path.join(METRICS_DIR, "training_and_inference_timings.csv")
timing_df = pd.read_csv(timing_csv_path) if os.path.exists(timing_csv_path) else pd.DataFrame()

for p_level in PHYSICS_LEVELS:
    for n_samples in DATA_TIERS:
        model_name = f"model_{p_level}_N{n_samples}"
        final_path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")

        if not os.path.exists(final_path):
            continue

        checkpoint = torch.load(final_path, map_location=device, weights_only=False)

        surrogate = FullyConnected(
            in_features=len(FEATURE_COLS),
            out_features=N_SPATIAL_POINTS,
            num_layers=6,
            layer_size=128,
        ).to(device)

        surrogate.load_state_dict(checkpoint["model_state_dict"])
        surrogate.eval()

        with torch.no_grad():
            Y_pred_norm = surrogate(X_test_tensor).cpu().numpy()

        Y_pred_nm = Y_pred_norm * Y_std + Y_mean
        all_predictions_dict[f"Y_pred_{model_name}"] = Y_pred_nm

        abs_err_matrix = np.abs(Y_test_nm - Y_pred_nm)
        sq_err_matrix = (Y_test_nm - Y_pred_nm) ** 2
        sample_mae = abs_err_matrix.mean(axis=1)

        global_r2 = float(r2_score(Y_test_nm.flatten(), Y_pred_nm.flatten()))
        global_mae = float(abs_err_matrix.mean())
        global_rmse = float(np.sqrt(sq_err_matrix.mean()))
        global_max_err = float(abs_err_matrix.max())
        global_mape = float(np.mean(abs_err_matrix / np.clip(Y_test_nm, 1e-6, None)) * 100.0)

        p10 = float(np.percentile(abs_err_matrix, 10))
        p25 = float(np.percentile(abs_err_matrix, 25))
        p50 = float(np.percentile(abs_err_matrix, 50))
        p75 = float(np.percentile(abs_err_matrix, 75))
        p90 = float(np.percentile(abs_err_matrix, 90))
        p95 = float(np.percentile(abs_err_matrix, 95))
        p99 = float(np.percentile(abs_err_matrix, 99))

        mae_fin_peak = float(abs_err_matrix[:, fin_peak_pts].mean())
        mae_sidewall = float(abs_err_matrix[:, sidewall_pts].mean())
        mae_trench_bottom = float(abs_err_matrix[:, trench_bottom_pts].mean())
        mae_corners = float(abs_err_matrix[:, corner_pts].mean())

        pred_top = Y_pred_nm[:, fin_peak_pts].mean(axis=1)
        pred_bot = Y_pred_nm[:, trench_bottom_pts].mean(axis=1)
        pred_conf = pred_bot / np.clip(pred_top, 1e-6, None)

        conf_r2 = float(r2_score(true_conf, pred_conf))
        conf_mae = float(mean_absolute_error(true_conf, pred_conf))
        conf_rmse = float(np.sqrt(mean_squared_error(true_conf, pred_conf)))

        corr_temp, _ = stats.pearsonr(df_test["temperature_C"].values, sample_mae)
        corr_pulse, _ = stats.pearsonr(df_test["pulse_time_s"].values, sample_mae)
        corr_cycles, _ = stats.pearsonr(df_test["num_cycles"].values, sample_mae)
        corr_tma, _ = stats.pearsonr(df_test["tma_mole_fraction"].values, sample_mae)
        corr_gpc, _ = stats.pearsonr(df_test["gpc_nm_per_cycle"].values, sample_mae)

        t_row = timing_df[timing_df["model_name"] == model_name]
        t_sec = float(t_row["total_train_time_seconds"].values[0]) if not t_row.empty else 0.0
        t_min = float(t_row["total_train_time_minutes"].values[0]) if not t_row.empty else 0.0
        throughput = float(t_row["inference_throughput_samples_per_sec"].values[0]) if not t_row.empty else 0.0
        latency_ms = float(t_row["single_sample_latency_ms"].values[0]) if not t_row.empty else 0.0
        vram_mb = float(t_row["peak_vram_allocated_mb"].values[0]) if not t_row.empty else 0.0
        file_kb = float(t_row["checkpoint_size_kb"].values[0]) if not t_row.empty else float(os.path.getsize(final_path)/1024)

        master_rows.append({
            "model_name": model_name,
            "physics_level": p_level,
            "training_samples": n_samples,
            "training_time_seconds": t_sec,
            "training_time_minutes": t_min,
            "inference_throughput_samples_per_sec": throughput,
            "single_sample_latency_ms": latency_ms,
            "peak_vram_allocated_mb": vram_mb,
            "checkpoint_size_kb": file_kb,
            "global_spatial_r2": global_r2,
            "global_spatial_mae_nm": global_mae,
            "global_spatial_rmse_nm": global_rmse,
            "global_spatial_max_error_nm": global_max_err,
            "global_spatial_mape_pct": global_mape,
            "error_p10_nm": p10,
            "error_p25_nm": p25,
            "error_p50_median_nm": p50,
            "error_p75_nm": p75,
            "error_p90_nm": p90,
            "error_p95_tail_nm": p95,
            "error_p99_extreme_nm": p99,
            "mae_fin_peak_nm": mae_fin_peak,
            "mae_sidewall_nm": mae_sidewall,
            "mae_trench_bottom_nm": mae_trench_bottom,
            "mae_corner_discontinuity_nm": mae_corners,
            "conformality_r2": conf_r2,
            "conformality_mae": conf_mae,
            "conformality_rmse": conf_rmse,
            "corr_err_vs_temp": float(corr_temp),
            "corr_err_vs_pulse": float(corr_pulse),
            "corr_err_vs_cycles": float(corr_cycles),
            "corr_err_vs_tma": float(corr_tma),
            "corr_err_vs_gpc": float(corr_gpc)
        })

        df_sample_master[f"pred_conf_{model_name}"] = pred_conf
        df_sample_master[f"sample_mae_{model_name}"] = sample_mae

        for pt_idx in range(N_SPATIAL_POINTS):
            pt_true = Y_test_nm[:, pt_idx]
            pt_pred = Y_pred_nm[:, pt_idx]
            pointwise_rows.append({
                "model_name": model_name,
                "physics_level": p_level,
                "training_samples": n_samples,
                "spatial_point_index": pt_idx,
                "coordinate_name": TARGET_COLS[pt_idx],
                "mae_nm": float(mean_absolute_error(pt_true, pt_pred)),
                "rmse_nm": float(np.sqrt(mean_squared_error(pt_true, pt_pred))),
                "r2_score": float(r2_score(pt_true, pt_pred)),
                "max_error_nm": float(np.max(np.abs(pt_true - pt_pred)))
            })

        tail_thresh = np.percentile(sample_mae, 95)
        tail_mask = sample_mae >= tail_thresh
        df_tail = df_test[tail_mask].copy()
        df_tail["sample_mae_nm"] = sample_mae[tail_mask]
        df_tail["peak_error_point_index"] = abs_err_matrix[tail_mask].argmax(axis=1)
        df_tail.to_csv(os.path.join(TAILS_DIR, f"outliers_5pct_{model_name}.csv"), index=False)

npz_path = os.path.join(RAW_PREDS_DIR, "all_models_test_predictions_2000x100.npz")
np.savez_compressed(npz_path, **all_predictions_dict)

df_master = pd.DataFrame(master_rows)
df_master.to_csv(os.path.join(TABLES_DIR, "master_research_metrics_all_16_models.csv"), index=False)

with open(os.path.join(TABLES_DIR, "master_research_metrics_all_16_models.json"), "w") as f:
    json.dump(master_rows, f, indent=4)

pd.DataFrame(pointwise_rows).to_csv(os.path.join(POINTWISE_DIR, "pointwise_spatial_profiles_all_16_models.csv"), index=False)
df_sample_master.to_csv(os.path.join(TABLES_DIR, "per_sample_test_predictions_master.csv"), index=False)

storage_audit_rows = []
for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        f_path = os.path.join(root, file)
        f_bytes = os.path.getsize(f_path)
        storage_audit_rows.append({
            "relative_path": os.path.relpath(f_path, BASE_PATH),
            "filename": file,
            "folder": os.path.basename(root),
            "size_bytes": f_bytes,
            "size_kb": round(f_bytes / 1024, 2),
            "size_mb": round(f_bytes / (1024**2), 4)
        })

df_audit = pd.DataFrame(storage_audit_rows)
audit_csv_path = os.path.join(TABLES_DIR, "master_drive_storage_file_size_audit.csv")
df_audit.to_csv(audit_csv_path, index=False)

latex_accuracy_table = r"""\begin{table*}[t]
\centering
\caption{Predictive Accuracy of Data-Driven MLPs vs. Physics-Informed Neural Networks Across Data Tiers ($N_{\text{test}} = 2,000$).}
\label{tab:pinn_accuracy_ablation}
\begin{tabular}{lcccccc}
\hline
\textbf{Model Framework} & \textbf{Data ($N$)} & \textbf{Global $R^2$} & \textbf{Global MAE (nm)} & \textbf{RMSE (nm)} & \textbf{Conf. $R^2$} & \textbf{95th \% Tail (nm)} \\
\hline
"""

for _, r in df_master.iterrows():
    p_name = {"P0": "Pure Data MLP ($P_0$)", "P1": "Boundary PINN ($P_1$)", "P2": "Kinetics PINN ($P_2$)", "P3": "Multiscale PINN ($P_3$)"}[r["physics_level"]]
    latex_accuracy_table += f"{p_name} & {int(r['training_samples'])} & {r['global_spatial_r2']:.6f} & {r['global_spatial_mae_nm']:.4e} & {r['global_spatial_rmse_nm']:.4e} & {r['conformality_r2']:.6f} & {r['error_p95_tail_nm']:.4e} \\\\\n"

latex_accuracy_table += r"""\hline
\end{tabular}
\end{table*}
"""

latex_efficiency_table = r"""\begin{table}[h]
\centering
\caption{Computational Resource Footprint and Inference Latency Across Surrogate Models.}
\label{tab:pinn_computational_footprint}
\begin{tabular}{lccccc}
\hline
\textbf{Model} & \textbf{Train Time (s)} & \textbf{Train Time (min)} & \textbf{Throughput (sam/s)} & \textbf{Latency (ms)} & \textbf{Size (KB)} \\
\hline
"""

for _, r in df_master.iterrows():
    latex_efficiency_table += f"{r['model_name']} & {r['training_time_seconds']:.2f} & {r['training_time_minutes']:.2f} & {r['inference_throughput_samples_per_sec']:.1f} & {r['single_sample_latency_ms']:.4f} & {r['checkpoint_size_kb']:.1f} \\\\\n"

latex_efficiency_table += r"""\hline
\end{tabular}
\end{table}
"""

latex_tex_path = os.path.join(TABLES_DIR, "latex_tables_for_paper.tex")
with open(latex_tex_path, "w") as f:
    f.write(latex_accuracy_table + "\n\n" + latex_efficiency_table)

print(f"Master Metrics CSV Saved : {os.path.join(TABLES_DIR, 'master_research_metrics_all_16_models.csv')}")
print(f"Storage File Audit Saved : {audit_csv_path} ({len(df_audit)} total drive files logged)")
print(f"LaTeX Tables Exported    : {latex_tex_path}")
